# Bot-IoT Version2 TOP10：分開分類 + 資料平衡比較

本筆記本分成上下兩段：
1. 上半段：直接使用原始資料做分類（不特別平衡）
2. 下半段：只對訓練集做 Data Balance（以 SMOTE 為主）後，再跑一次同樣分類流程

每一種分類法都會獨立執行，並在執行前用中文說明模型用途與參數設定。

## 流程總覽

前置作業 -> 各種分類法（原始資料） -> 視覺化結果1 -> Data Balance（僅訓練集） -> 各種分類法（平衡資料） -> 視覺化結果2

另外加入可選的交叉驗證（5-fold / 10-fold），方便報告說明是否有交叉比對。

In [ ]:
import warnings
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)
from xgboost import XGBClassifier

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable, **kwargs):
        return iterable

try:
    from imblearn.over_sampling import SMOTE, RandomOverSampler
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

## 參數與切割設定（報告可直接引用）

- 資料來源：`Dataset_for_train_and_test` 中官方 Training / Testing 檔
- 特徵：論文 TOP10（10 個數值特徵）
- 目標：`attack`、`category`、`subcategory`
- 訓練/驗證切割：從 Training 檔再切出 `25% validation`，使用三標籤組合分層抽樣（避免分布偏移）
- 測試：最後使用官方 Testing 檔做最終評估
- 交叉驗證：提供 5-fold / 10-fold 選項（預設關閉，可在下方開關）

In [ ]:
RANDOM_STATE = 42
TRAIN_PATH = "Dataset_for_train_and_test/UNSW_2018_IoT_Botnet_Final_10_best_Training.csv"
TEST_PATH = "Dataset_for_train_and_test/UNSW_2018_IoT_Botnet_Final_10_best_Testing.csv"

FEATURE_COLUMNS = [
    "seq",
    "stddev",
    "N_IN_Conn_P_SrcIP",
    "min",
    "state_number",
    "mean",
    "N_IN_Conn_P_DstIP",
    "drate",
    "srate",
    "max",
]
TARGET_COLUMNS = ["attack", "category", "subcategory"]

VALID_SIZE = 0.25
BALANCE_METHOD = "smote"  # smote 或 random_over

ENABLE_CV = False
CV_FOLDS = 5  # 可改 10
CV_SAMPLE_SIZE = 120000  # 避免運行時間過長
CHECKPOINT_DIR = "artifacts"

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

In [ ]:
X_all = train_df[FEATURE_COLUMNS].copy()
y_all = train_df[TARGET_COLUMNS].copy()
X_test_official = test_df[FEATURE_COLUMNS].copy()
y_test_official = test_df[TARGET_COLUMNS].copy()

label_encoders = {}
for target_col in ["category", "subcategory"]:
    le = LabelEncoder()
    y_all[target_col] = le.fit_transform(y_all[target_col])
    y_test_official[target_col] = le.transform(y_test_official[target_col])
    label_encoders[target_col] = le

stratify_key = (
    y_all["attack"].astype(str) + "_" +
    y_all["category"].astype(str) + "_" +
    y_all["subcategory"].astype(str)
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_all,
    y_all,
    test_size=VALID_SIZE,
    random_state=RANDOM_STATE,
    stratify=stratify_key
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_all_scaled = scaler.fit_transform(X_all)
X_test_official_scaled = scaler.transform(X_test_official)

print("X_train:", X_train.shape, "X_valid:", X_valid.shape)
print("分層切割 key 範例:", stratify_key.iloc[0])

In [ ]:
for col in TARGET_COLUMNS:
    print(f"\n{col} distribution (train split):")
    print(y_train[col].value_counts(normalize=True).head(10))

## 共用函式（訓練、評估、視覺化）

In [ ]:
class ChainedMultiTargetClassifier:
    def __init__(self, attack_model, category_model, subcategory_model):
        self.attack_model = attack_model
        self.category_model = category_model
        self.subcategory_model = subcategory_model

    def fit(self, X, y, show_progress=True, desc="model"):
        pbar = tqdm(total=3, desc=desc, unit="stage") if show_progress else None

        self.attack_model.fit(X, y["attack"])
        if pbar is not None:
            pbar.update(1)
            pbar.set_postfix_str("attack done")

        category_features = np.concatenate([X, y[["attack"]].to_numpy()], axis=1)
        self.category_model.fit(category_features, y["category"])
        if pbar is not None:
            pbar.update(1)
            pbar.set_postfix_str("category done")

        subcategory_features = np.concatenate([category_features, y[["category"]].to_numpy()], axis=1)
        self.subcategory_model.fit(subcategory_features, y["subcategory"])
        if pbar is not None:
            pbar.update(1)
            pbar.set_postfix_str("subcategory done")
            pbar.close()

        return self

    def predict(self, X):
        pred_attack = self.attack_model.predict(X)
        category_features = np.concatenate([X, pred_attack.reshape(-1, 1)], axis=1)
        pred_category = self.category_model.predict(category_features)
        subcategory_features = np.concatenate([category_features, pred_category.reshape(-1, 1)], axis=1)
        pred_subcategory = self.subcategory_model.predict(subcategory_features)
        return pd.DataFrame({
            "attack": pred_attack,
            "category": pred_category,
            "subcategory": pred_subcategory,
        })

def build_chain_model(model_name, seed=RANDOM_STATE):
    n_category = y_all["category"].nunique()
    n_subcategory = y_all["subcategory"].nunique()

    if model_name == "RandomForest":
        return ChainedMultiTargetClassifier(
            RandomForestClassifier(max_depth=8, n_estimators=120, random_state=seed, n_jobs=-1),
            RandomForestClassifier(max_depth=10, n_estimators=120, random_state=seed, n_jobs=-1),
            RandomForestClassifier(max_depth=12, n_estimators=120, random_state=seed, n_jobs=-1),
        )

    if model_name == "NaiveBayes":
        return ChainedMultiTargetClassifier(GaussianNB(), GaussianNB(), GaussianNB())

    if model_name == "DecisionTreeEntropy":
        return ChainedMultiTargetClassifier(
            DecisionTreeClassifier(criterion="entropy", max_depth=8, random_state=seed),
            DecisionTreeClassifier(criterion="entropy", max_depth=10, random_state=seed),
            DecisionTreeClassifier(criterion="entropy", max_depth=12, random_state=seed),
        )

    if model_name == "DecisionTreeGini":
        return ChainedMultiTargetClassifier(
            DecisionTreeClassifier(criterion="gini", max_depth=8, random_state=seed),
            DecisionTreeClassifier(criterion="gini", max_depth=10, random_state=seed),
            DecisionTreeClassifier(criterion="gini", max_depth=12, random_state=seed),
        )

    if model_name == "XGBoost":
        return ChainedMultiTargetClassifier(
            XGBClassifier(
                objective="binary:logistic",
                n_estimators=150,
                max_depth=6,
                learning_rate=0.08,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=seed,
                eval_metric="logloss",
                n_jobs=-1,
            ),
            XGBClassifier(
                objective="multi:softmax",
                num_class=n_category,
                n_estimators=150,
                max_depth=6,
                learning_rate=0.08,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=seed,
                eval_metric="mlogloss",
                n_jobs=-1,
            ),
            XGBClassifier(
                objective="multi:softmax",
                num_class=n_subcategory,
                n_estimators=150,
                max_depth=6,
                learning_rate=0.08,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=seed,
                eval_metric="mlogloss",
                n_jobs=-1,
            ),
        )

    raise ValueError(f"Unknown model: {model_name}")

def evaluate_predictions(y_true, y_pred, target_col):
    y_t = y_true[target_col]
    y_p = y_pred[target_col]
    return {
        "target": target_col,
        "accuracy": accuracy_score(y_t, y_p),
        "balanced_accuracy": balanced_accuracy_score(y_t, y_p),
        "macro_f1": f1_score(y_t, y_p, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_t, y_p, average="weighted", zero_division=0),
    }

def save_metrics_checkpoint(metrics_df, section_tag, model_name):
    Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
    out_path = Path(CHECKPOINT_DIR) / f"{section_tag}_{model_name}_metrics.csv"
    metrics_df.to_csv(out_path, index=False)
    print(f"checkpoint saved: {out_path}")

def run_single_model(model_name, X_fit, y_fit, X_eval, y_eval, section_tag="section"):
    model = build_chain_model(model_name)
    t0 = time.time()
    model.fit(X_fit, y_fit, show_progress=True, desc=f"{section_tag}:{model_name}")
    preds = model.predict(X_eval)

    rows = []
    for target_col in TARGET_COLUMNS:
        row = evaluate_predictions(y_eval, preds, target_col)
        row["model"] = model_name
        rows.append(row)

    metrics_df = pd.DataFrame(rows)
    save_metrics_checkpoint(metrics_df, section_tag, model_name)
    print(f"{section_tag}:{model_name} finished in {(time.time() - t0)/60:.2f} min")

    return model, preds, metrics_df

def decode_labels_if_needed(values, target_col):
    if target_col in label_encoders:
        return label_encoders[target_col].inverse_transform(values)
    return values

def show_confusion_matrices(result_store, y_true, section_title):
    metrics_df = pd.concat([v["metrics"] for v in result_store.values()], ignore_index=True)

    best_map = {}
    for target_col in TARGET_COLUMNS:
        picked = metrics_df[metrics_df["target"] == target_col].sort_values("macro_f1", ascending=False).iloc[0]
        best_map[target_col] = picked["model"]

    print(f"{section_title} 各目標最佳模型：", best_map)

    for target_col in TARGET_COLUMNS:
        best_model = best_map[target_col]
        y_t = y_true[target_col].to_numpy()
        y_p = result_store[best_model]["preds"][target_col].to_numpy()

        labels = np.unique(np.concatenate([y_t, y_p]))
        cm = confusion_matrix(y_t, y_p, labels=labels)

        tick_labels = decode_labels_if_needed(labels, target_col)

        plt.figure(figsize=(7, 5))
        sns.heatmap(cm, cmap="Blues", annot=False, fmt="d")
        plt.title(f"{section_title} | {target_col} | Best: {best_model}")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.xticks(np.arange(len(tick_labels)) + 0.5, tick_labels, rotation=45, ha="right")
        plt.yticks(np.arange(len(tick_labels)) + 0.5, tick_labels, rotation=0)
        plt.tight_layout()
        plt.show()

def show_metric_bars(metrics_df, title_prefix):
    for target_col in TARGET_COLUMNS:
        sub_df = metrics_df[metrics_df["target"] == target_col].sort_values("macro_f1", ascending=False)
        plt.figure(figsize=(8, 4))
        sns.barplot(data=sub_df, x="model", y="macro_f1", color="#4E79A7")
        plt.ylim(0, 1)
        plt.title(f"{title_prefix} | {target_col} Macro-F1")
        plt.xticks(rotation=25)
        plt.tight_layout()
        plt.show()

def build_attack_model(model_name, seed=RANDOM_STATE):
    if model_name == "RandomForest":
        return RandomForestClassifier(max_depth=8, n_estimators=120, random_state=seed, n_jobs=-1)
    if model_name == "NaiveBayes":
        return GaussianNB()
    if model_name == "DecisionTreeEntropy":
        return DecisionTreeClassifier(criterion="entropy", max_depth=8, random_state=seed)
    if model_name == "DecisionTreeGini":
        return DecisionTreeClassifier(criterion="gini", max_depth=8, random_state=seed)
    if model_name == "XGBoost":
        return XGBClassifier(
            objective="binary:logistic",
            n_estimators=120,
            max_depth=6,
            learning_rate=0.08,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=seed,
            eval_metric="logloss",
            n_jobs=-1,
        )
    raise ValueError(f"Unknown model: {model_name}")

def run_attack_cv(X, y_attack, model_names, folds=5, sample_size=120000):
    sample_size = min(sample_size, len(y_attack))
    idx = np.random.RandomState(RANDOM_STATE).choice(len(y_attack), size=sample_size, replace=False)
    X_sub = X[idx]
    y_sub = y_attack.iloc[idx].to_numpy()

    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=RANDOM_STATE)
    rows = []

    for name in tqdm(model_names, desc=f"CV {folds}-fold", unit="model"):
        est = build_attack_model(name)
        scores = cross_validate(
            est,
            X_sub,
            y_sub,
            cv=skf,
            scoring={
                "acc": "accuracy",
                "bacc": "balanced_accuracy",
                "f1_macro": "f1_macro",
            },
            n_jobs=1,
            return_train_score=False,
        )

        rows.append({
            "model": name,
            "folds": folds,
            "sample_size": sample_size,
            "cv_accuracy_mean": scores["test_acc"].mean(),
            "cv_balanced_accuracy_mean": scores["test_bacc"].mean(),
            "cv_macro_f1_mean": scores["test_f1_macro"].mean(),
        })

    return pd.DataFrame(rows).sort_values("cv_macro_f1_mean", ascending=False)

# 上半部：原始資料（不做 Data Balance）

這一段先建立基準結果，模擬『不考慮資料平衡』的傳統分類流程。

### 分類法 1：Random Forest
- 用途：透過多棵樹集成，降低單一決策樹過擬合風險
- 參數：`max_depth=[8,10,12]`、`n_estimators=120`、`random_state=42`

In [ ]:
unbalanced_results = {}

rf_model, rf_preds, rf_metrics = run_single_model("RandomForest", X_train_scaled, y_train, X_valid_scaled, y_valid, section_tag="unbalanced")
unbalanced_results["RandomForest"] = {"model": rf_model, "preds": rf_preds, "metrics": rf_metrics}
rf_metrics

### 分類法 2：Naive Bayes
- 用途：速度快、可做機率式分類基準
- 參數：`GaussianNB()`（使用預設參數）

In [ ]:
nb_model, nb_preds, nb_metrics = run_single_model("NaiveBayes", X_train_scaled, y_train, X_valid_scaled, y_valid, section_tag="unbalanced")
unbalanced_results["NaiveBayes"] = {"model": nb_model, "preds": nb_preds, "metrics": nb_metrics}
nb_metrics

### 分類法 3：Decision Tree (Entropy)
- 用途：資訊增益（Entropy）切分規則
- 參數：`criterion=entropy`、`max_depth=[8,10,12]`

In [ ]:
dt_e_model, dt_e_preds, dt_e_metrics = run_single_model("DecisionTreeEntropy", X_train_scaled, y_train, X_valid_scaled, y_valid, section_tag="unbalanced")
unbalanced_results["DecisionTreeEntropy"] = {"model": dt_e_model, "preds": dt_e_preds, "metrics": dt_e_metrics}
dt_e_metrics

### 分類法 4：Decision Tree (Gini)
- 用途：Gini impurity 切分規則
- 參數：`criterion=gini`、`max_depth=[8,10,12]`

In [ ]:
dt_g_model, dt_g_preds, dt_g_metrics = run_single_model("DecisionTreeGini", X_train_scaled, y_train, X_valid_scaled, y_valid, section_tag="unbalanced")
unbalanced_results["DecisionTreeGini"] = {"model": dt_g_model, "preds": dt_g_preds, "metrics": dt_g_metrics}
dt_g_metrics

### 分類法 5：XGBoost
- 用途：梯度提升樹，通常在結構化資料有較好表現
- 參數：`n_estimators=150`、`max_depth=6`、`learning_rate=0.08`、`subsample=0.8`、`colsample_bytree=0.8`

In [ ]:
xgb_model, xgb_preds, xgb_metrics = run_single_model("XGBoost", X_train_scaled, y_train, X_valid_scaled, y_valid, section_tag="unbalanced")
unbalanced_results["XGBoost"] = {"model": xgb_model, "preds": xgb_preds, "metrics": xgb_metrics}
xgb_metrics

### 原始資料結果整理 + 視覺化結果1
- 顯示各模型指標表
- 畫出 Macro-F1 條圖
- 顯示每個目標最佳模型的混淆矩陣

In [ ]:
unbalanced_metrics_df = pd.concat([v["metrics"] for v in unbalanced_results.values()], ignore_index=True)
display(unbalanced_metrics_df.sort_values(["target", "macro_f1"], ascending=[True, False]))

show_metric_bars(unbalanced_metrics_df, "Unbalanced Validation")
show_confusion_matrices(unbalanced_results, y_valid, "Unbalanced Validation")

print("
Detailed report (validation) for XGBoost / category:")
print(classification_report(y_valid["category"], unbalanced_results["XGBoost"]["preds"]["category"], zero_division=0))

### 交叉驗證（可選）
- 用途：回應報告中『是否做 5/10 折交叉比對』
- 設計：先用 `attack` 目標做模型穩定度檢查（可切換 `CV_FOLDS=5 或 10`）

In [ ]:
MODEL_NAMES = ["RandomForest", "NaiveBayes", "DecisionTreeEntropy", "DecisionTreeGini", "XGBoost"]

if ENABLE_CV:
    cv_result_unbalanced = run_attack_cv(X_train_scaled, y_train["attack"], MODEL_NAMES, folds=CV_FOLDS, sample_size=CV_SAMPLE_SIZE)
    display(cv_result_unbalanced)
else:
    print("交叉驗證目前關閉。若報告需要，將 ENABLE_CV=True，並設定 CV_FOLDS=5 或 10。")

# 下半部：Data Balance 後再分類

這一段只對訓練資料做平衡化，驗證集維持原分布，避免資料洩漏。

### 這裡採用的 SMOTE 設計（重點）
- 不再使用 `attack|category|subcategory` 三層合併做 SMOTE，避免類別碎裂與記憶體暴增。
- 只以 `subcategory` 當作重採樣目標，因為它已包含更細粒度的攻擊型態資訊。
- 極少數樣本（少於 6 筆）先用 `RandomOverSampler`；其餘類別再用 `SMOTE(k_neighbors<=5)`。
- 設定採樣上限（cap）避免把所有類別補到最大類，降低 RAM 壓力並保留分布特性。
- 若環境沒有 `imblearn`，則 fallback 到手動隨機過採樣。

In [ ]:
def balance_train_data(X_train_arr, y_train_df, method="smote", seed=RANDOM_STATE):
    # 關鍵修正：避免 y 的舊 index 與 X 的 numpy 位置索引不一致
    y_train_df = y_train_df.reset_index(drop=True).copy()

    # 以 subcategory 做重採樣，避免三層合併造成類別爆炸
    y_sub = y_train_df["subcategory"].astype(int).reset_index(drop=True)

    # 建立 subcategory -> attack/category 對應（Bot-IoT 層級標籤應為一對一）
    map_attack = y_train_df.groupby("subcategory")["attack"].agg(lambda s: s.mode().iloc[0]).to_dict()
    map_category = y_train_df.groupby("subcategory")["category"].agg(lambda s: s.mode().iloc[0]).to_dict()

    class_counts = y_sub.value_counts()
    max_count = int(class_counts.max())
    cap_target = int(max_count * 0.25)  # 上限: 最高類別的 25%

    target_count = {}
    for cls, cnt in class_counts.items():
        target_count[cls] = int(min(max(cnt, min(cap_target, max_count)), cap_target))

    tiny_classes = class_counts[class_counts < 6].index.tolist()
    normal_classes = class_counts[class_counts >= 6].index.tolist()

    if HAS_IMBLEARN:
        # 先補 tiny class（SMOTE 無法處理過少樣本）
        if len(tiny_classes) > 0:
            ros_strategy = {c: target_count[c] for c in tiny_classes if target_count[c] > class_counts[c]}
            if len(ros_strategy) > 0:
                ros = RandomOverSampler(random_state=seed, sampling_strategy=ros_strategy)
                X_step, y_step = ros.fit_resample(X_train_arr, y_sub)
            else:
                X_step, y_step = X_train_arr, y_sub
        else:
            X_step, y_step = X_train_arr, y_sub

        if method == "smote" and len(normal_classes) > 0:
            counts_after_ros = pd.Series(y_step).value_counts()
            smote_candidates = [c for c in normal_classes if counts_after_ros.get(c, 0) >= 6]
            smote_strategy = {c: target_count[c] for c in smote_candidates if target_count[c] > counts_after_ros.get(c, 0)}

            if len(smote_strategy) > 0:
                min_count = min(counts_after_ros[c] for c in smote_strategy.keys())
                k_neighbors = int(min(5, min_count - 1))
                sm = SMOTE(random_state=seed, k_neighbors=k_neighbors, sampling_strategy=smote_strategy)
                X_res, y_res_sub = sm.fit_resample(X_step, y_step)
                used_method = f"Hybrid(ROS tiny + SMOTE subcategory, k={k_neighbors}, cap=25%)"
            else:
                X_res, y_res_sub = X_step, y_step
                used_method = "ROS only (no eligible class for SMOTE)"
        else:
            X_res, y_res_sub = X_step, y_step
            used_method = "RandomOverSampler only"

        y_res_sub = pd.Series(y_res_sub).astype(int).reset_index(drop=True)

    else:
        # imblearn 不存在時的安全 fallback（位置索引，避免越界）
        used_method = "ManualRandomOversampling(subcategory, cap=25%)"
        rng = np.random.RandomState(seed)
        class_to_pos = {cls: np.flatnonzero(y_sub.to_numpy() == cls) for cls in class_counts.index}

        sampled_positions = []
        for cls, pos_arr in class_to_pos.items():
            desired = target_count[cls]
            if desired <= len(pos_arr):
                picked = rng.choice(pos_arr, size=desired, replace=False)
            else:
                picked = rng.choice(pos_arr, size=desired, replace=True)
            sampled_positions.append(picked)

        final_idx = np.concatenate(sampled_positions)
        rng.shuffle(final_idx)

        X_res = X_train_arr[final_idx]
        y_res_sub = y_sub.iloc[final_idx].reset_index(drop=True)

    y_res = pd.DataFrame({
        "subcategory": y_res_sub.astype(int),
    })
    y_res["attack"] = y_res["subcategory"].map(map_attack).astype(int)
    y_res["category"] = y_res["subcategory"].map(map_category).astype(int)
    y_res = y_res[["attack", "category", "subcategory"]]

    return X_res, y_res, used_method

X_train_balanced, y_train_balanced, used_balance_method = balance_train_data(
    X_train_scaled, y_train, method=BALANCE_METHOD, seed=RANDOM_STATE
)

print("Balance method:", used_balance_method)
print("Before:", X_train_scaled.shape, "After:", X_train_balanced.shape)

In [ ]:
for col in TARGET_COLUMNS:
    print(f"\nBalanced {col} distribution:")
    print(y_train_balanced[col].value_counts(normalize=True).head(10))

### 分類法 1（Balance）：Random Forest
- 用途：和上半部同模型，觀察平衡化後差異
- 參數：`max_depth=[8,10,12]`、`n_estimators=120`

In [ ]:
balanced_results = {}

rf_model_b, rf_preds_b, rf_metrics_b = run_single_model("RandomForest", X_train_balanced, y_train_balanced, X_valid_scaled, y_valid, section_tag="balanced")
balanced_results["RandomForest"] = {"model": rf_model_b, "preds": rf_preds_b, "metrics": rf_metrics_b}
rf_metrics_b

### 分類法 2（Balance）：Naive Bayes
- 用途：觀察機率模型在平衡後對少數類別是否改善
- 參數：`GaussianNB()`

In [ ]:
nb_model_b, nb_preds_b, nb_metrics_b = run_single_model("NaiveBayes", X_train_balanced, y_train_balanced, X_valid_scaled, y_valid, section_tag="balanced")
balanced_results["NaiveBayes"] = {"model": nb_model_b, "preds": nb_preds_b, "metrics": nb_metrics_b}
nb_metrics_b

### 分類法 3（Balance）：Decision Tree (Entropy)
- 用途：比較 entropy tree 在平衡後的穩定性
- 參數：`criterion=entropy`、`max_depth=[8,10,12]`

In [ ]:
dt_e_model_b, dt_e_preds_b, dt_e_metrics_b = run_single_model("DecisionTreeEntropy", X_train_balanced, y_train_balanced, X_valid_scaled, y_valid, section_tag="balanced")
balanced_results["DecisionTreeEntropy"] = {"model": dt_e_model_b, "preds": dt_e_preds_b, "metrics": dt_e_metrics_b}
dt_e_metrics_b

### 分類法 4（Balance）：Decision Tree (Gini)
- 用途：比較 gini tree 在平衡後對少數類別影響
- 參數：`criterion=gini`、`max_depth=[8,10,12]`

In [ ]:
dt_g_model_b, dt_g_preds_b, dt_g_metrics_b = run_single_model("DecisionTreeGini", X_train_balanced, y_train_balanced, X_valid_scaled, y_valid, section_tag="balanced")
balanced_results["DecisionTreeGini"] = {"model": dt_g_model_b, "preds": dt_g_preds_b, "metrics": dt_g_metrics_b}
dt_g_metrics_b

### 分類法 5（Balance）：XGBoost
- 用途：比較梯度提升樹在平衡後整體 F1 是否改善
- 參數：`n_estimators=150`、`max_depth=6`、`learning_rate=0.08`

In [ ]:
xgb_model_b, xgb_preds_b, xgb_metrics_b = run_single_model("XGBoost", X_train_balanced, y_train_balanced, X_valid_scaled, y_valid, section_tag="balanced")
balanced_results["XGBoost"] = {"model": xgb_model_b, "preds": xgb_preds_b, "metrics": xgb_metrics_b}
xgb_metrics_b

### 平衡資料結果整理 + 視覺化結果2
- 顯示各模型指標表
- 畫出 Macro-F1 條圖
- 顯示每個目標最佳模型的混淆矩陣

In [ ]:
balanced_metrics_df = pd.concat([v["metrics"] for v in balanced_results.values()], ignore_index=True)
display(balanced_metrics_df.sort_values(["target", "macro_f1"], ascending=[True, False]))

show_metric_bars(balanced_metrics_df, "Balanced Validation")
show_confusion_matrices(balanced_results, y_valid, "Balanced Validation")

### 交叉驗證（Balance，可選）
- 和上半部相同設定，用於比較平衡前後穩定度

In [ ]:
if ENABLE_CV:
    cv_result_balanced = run_attack_cv(X_train_balanced, y_train_balanced["attack"], MODEL_NAMES, folds=CV_FOLDS, sample_size=CV_SAMPLE_SIZE)
    display(cv_result_balanced)
else:
    print("交叉驗證目前關閉。若報告需要，將 ENABLE_CV=True，並設定 CV_FOLDS=5 或 10。")

## 最終測試集比較（可放報告總表）

這裡示範用原始資料與平衡資料，各自重新在完整 training 上訓練，再到官方 testing 測試。

In [ ]:
Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
final_rows = []

for name in tqdm(MODEL_NAMES, desc="Final testing", unit="model"):
    model_unbal = build_chain_model(name)
    model_unbal.fit(X_all_scaled, y_all)
    pred_unbal = model_unbal.predict(X_test_official_scaled)

    X_all_bal, y_all_bal, _ = balance_train_data(X_all_scaled, y_all, method=BALANCE_METHOD, seed=RANDOM_STATE)
    model_bal = build_chain_model(name)
    model_bal.fit(X_all_bal, y_all_bal)
    pred_bal = model_bal.predict(X_test_official_scaled)

    for target_col in TARGET_COLUMNS:
        r1 = evaluate_predictions(y_test_official, pred_unbal, target_col)
        r1["model"] = name
        r1["setting"] = "unbalanced_train"

        r2 = evaluate_predictions(y_test_official, pred_bal, target_col)
        r2["model"] = name
        r2["setting"] = "balanced_train"

        final_rows.extend([r1, r2])

    pd.DataFrame(final_rows).to_csv(Path(CHECKPOINT_DIR) / "final_test_partial.csv", index=False)

final_test_result = pd.DataFrame(final_rows)
display(final_test_result.sort_values(["target", "setting", "macro_f1"], ascending=[True, True, False]))